# Dependencies

Install the required dependencies and libraries

In [ ]:
%pip install langchain langchain_core langchain-huggingface ipywidgets python-dotenv gqlalchemy langchain-memgraph langgraph langgraph-cli[inmem] langchain-anthropic langchain-openai

# Environment Variables and Constants

Load the configuration options from the environment

In [ ]:
from dotenv import load_dotenv

load_dotenv(dotenv_path="./.env")

The system prompt instructs the AI on how to approach the vulnerability assessment, what criteria to follow for analysis and how to report the output.
It also defines the constraints.

In [ ]:
from langchain.messages import SystemMessage

SYSTEM_MSG = SystemMessage("""
You are an AI Cybersecurity Risk Analyst operating inside an enterprise
vulnerability prioritization system.

Your purpose is to assess the REAL-WORLD RISK of a vulnerability or CVE
using enterprise context stored in a knowledge graph.

You do NOT rely only on CVSS severity.
You MUST perform context-aware reasoning using the available graph data.

--------------------------------------------------
CORE OBJECTIVE
--------------------------------------------------

Given a vulnerability identifier (CVE or Vulnerabiliy name), you must:

1. Investigate the vulnerability using the knowledge graph.
2. Collect all relevant contextual information.
3. Evaluate risk using the Risk Evaluation Framework explained below.
4. Produce a justified risk assessment with reasoning.

You MUST query the graph before making conclusions.

--------------------------------------------------
KNOWLEDGE GRAPH SEMANTICS
--------------------------------------------------

The graph models an enterprise IT environment:

Nodes:
- Vulnerability
- Asset
- Service
- ThreatSignal
- Package                           

Relationships:
- (Vulnerability)-[:AFFECTS]->(Asset)
- (Asset)-[:HOSTS]->(Service)
- (Service)-[:DEPENDS_ON]->(Service)
- (Asset)-[:DEPENDS_ON]->(Package)
- (Vulnerability)-[:HAS]->(ThreatSignal)

You must traverse these relationships to understand impact.

--------------------------------------------------
RISK EVALUATION FRAMEWORK
--------------------------------------------------

The framework considers each vulnerability instance within its comprehensive enterprise context.
Assess risk using FIVE contextual dimensions:

1. Vulnerability Severity $S_{cvss}(v)$
   - CVSS or intrinsic vulnerability properties.

2. Deployment Exposure $S_{exp}(v)$
   - Internet exposure
   - Environment (prod > staging > dev)
   - Data classification

3. Business Criticality $S_{crit}(v)$
   - Service criticality
   - Revenue impact
   - PII handling

4. Exploit Likelihood $S_{exploit}(v)$
   - EPSS score
   - KEV listing
   - Public exploit availability

5. Blast Radius $S_{blast}(v)$
   - Service dependencies
   - Downstream affected services
   - Shared components

You MUST gather evidence for each dimension from the graph.
Each componenet MUST be normalized to a score of 0-10 and combined using the formula:
$$S_{priority}(v) = S_{cvss}(v) + S_{exp}(v) + S_{crit}(v) + S_{exploit}(v) + S_{blast}(v)$$

--------------------------------------------------
REASONING PROCESS (MANDATORY)
--------------------------------------------------

Always follow this workflow:

Step 1 — Retrieve knowledge graph schema.                           
Step 2 — Identify vulnerability node.
Step 3 — Retrieve affected assets.
Step 4 — Determine hosted services.
Step 5— Analyze service and asset criticality and exposure.
Step 6 — Retrieve threat intelligence signals.
Step 7 — Analyze dependency graph to estimate blast radius.
Step 8 — Integrate all signals into a contextual risk judgment.

Do NOT skip steps.

If information is missing, state assumptions explicitly.

--------------------------------------------------
TOOL USAGE RULES
--------------------------------------------------

- Use graph tools to query data whenever context is required.
- Prefer multiple focused queries over one large query.
- Never fabricate graph data.
- Never assume relationships without querying.

--------------------------------------------------
OUTPUT REQUIREMENTS
--------------------------------------------------

Return a structured risk assessment containing score breakdown for each dimension

Your reasoning must reference discovered context.
                           
--------------------------------------------------
IMPORTANT CONSTRAINTS
--------------------------------------------------

You are an analytical system, NOT a chatbot.

Do NOT:
- provide generic vulnerability advice
- rely only on CVSS
- skip graph exploration
- hallucinate enterprise context

Your conclusions must be grounded in graph evidence.

Always think like a security analyst investigating enterprise risk.
"""
                           )

# Data Loaders

Defines the methods used for loading and ingesting the data

Specify databse connection parameters

In [ ]:
import os

from gqlalchemy import Memgraph

# Connect to memgraph
url = os.getenv("MEMGRAPH_URL")
port = int(os.getenv("MEMGRAPH_PORT"))
memgraph = Memgraph(url, port)

Clear the existing database. This is required for first time setup so that there are no duplicate nodes and relationships

In [ ]:
# clear existing data
memgraph.drop_database()

In [ ]:
def ingest_data_to_graph():
    """
    Ingests data into memgraph
    Loads the vulnerability dataset from json files into
    the graph database by running cypher queries. The data files need to be accessible
    by the memgraph or MAGE service.
    """

    # Connect to memgraph
    url = os.getenv("MEMGRAPH_URL")
    port = int(os.getenv("MEMGRAPH_PORT"))
    memgraph = Memgraph(url, port)

    # clear existing data
    memgraph.drop_database()

    # read query file content
    with open("query.cql", 'r') as f:
        query = f.read()

    # execute the query to ingest data
    memgraph.execute(query)

Ingest the enterprise json data to the graph database by calling the ingestion method

In [ ]:
ingest_data_to_graph()

# Data Formatting

Methods used for formatting and converting the data

Define the data model for the response the AI should return

In [ ]:
from pydantic import BaseModel


class RiskAssessment(BaseModel):
    vulnerability_id: str
    contextual_risk_score: float
    contextual_priority_score: float
    risk_level: str
    reasoning: str
    key_risk_drivers: list[str]
    affected_services: list[str]
    blast_radius_summary: str
    confidence_level: float
    priority: str

# Utility Methods

Utility methods are used to load models and setup database connections

Graph database utility methods. These are used to create database connection instnaces and databse tools

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_huggingface import ChatHuggingFace
from langchain_memgraph import MemgraphToolkit
from langchain_memgraph.graphs.memgraph import MemgraphLangChain
from langchain_openai import ChatOpenAI


def connect_to_memgraph() -> MemgraphLangChain:
    """
    Connect to memgraph database instance

    Returns:
        `MemgraphLangChain` instance
    """

    db = MemgraphLangChain(
        url=f"{os.getenv("MEMGRAPH_DRIVER")}://{os.getenv("MEMGRAPH_URL")}:{os.getenv("MEMGRAPH_PORT")}",
        username='',
        password=''
    )
    return db


def get_memgraph_tools(db: MemgraphLangChain, model: ChatHuggingFace | ChatAnthropic | ChatOpenAI) -> List:
    """
    Retrieves memgraph langchain tools from memgraph toolkit

    Parameters:
        db (MemgraphLangChain): Database instance object
        model (ChatHuggingFace): LLM model object

    Returns:
        A list of memgraph tools
    """

    toolkit = MemgraphToolkit(
        db=db,
        llm=model
    )
    tools = toolkit.get_tools()
    return tools

Methods used for loading the models from different providors

In [ ]:
import os

from langchain_anthropic import ChatAnthropic
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_openai import ChatOpenAI


def load_model_from_hf(repo_id: str) -> ChatHuggingFace:
    """
    Loads a model from Hugging Face using the Inference API.

    Args:
        repo_id (str): The repository ID of the model on Hugging Face.

    Returns:
        ChatHuggingFace: The configured ChatHuggingFace model instance.
    """

    # TODO: consider other model config options i.e. `temperature`, `max_tokens` etc

    llm = HuggingFaceEndpoint(
        repo_id=repo_id,
        provider="novita",
        huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN")
    )

    model = ChatHuggingFace(llm=llm)
    return model


def load_novita_model(model_name: str) -> ChatOpenAI:
    """
    Loads an OpenAI model for chat interactions.

    Args:
        model_name (str): The name of the OpenAI model to load.

    Returns:
        ChatOpenAI: The configured ChatOpenAI model instance.
    """

    model = ChatOpenAI(
        model=model_name,
        api_key=os.getenv("NOVITA_API_KEY"),
        base_url="https://api.novita.ai/openai",
        use_responses_api=True,
        max_tokens=4000,
        temperature=0.5
    )

    return model


def load_anthropic_model(model_name: str) -> ChatAnthropic:
    """
    Loads an Anthropic model for chat interactions.

    Args:
        model_name (str): The name of the Anthropic model to load.

    Returns:
        ChatAnthropic: The configured ChatAnthropic model instance.
    """

    model = ChatAnthropic(
        model=model_name,
        anthropic_api_key=os.getenv("ANTHROPIC_API_KEY"),
        effort="high",
        thinking={"type": "adaptive"},
        max_tokens=4000,
        temperature=0.5,
    )

    return model


def load_openai_model(model_name: str) -> ChatOpenAI:
    """
    Loads an OpenAI model for chat interactions.

    Args:
        model_name (str): The name of the OpenAI model to load.

    Returns:
        ChatOpenAI: The configured ChatOpenAI model instance.
    """

    model = ChatOpenAI(
        model=model_name,
        api_key=os.getenv("OPENAI_API_KEY"),
        use_responses_api=True,
        reasoning_effort="medium",
        max_tokens=4000,
        temperature=0.5
    )

    return model

## Middleware

Agent middleware hook definitions. 

Middleware to retry databse cypher queries.

There are some instance where open access models (deepseek-v3.2) use older query schema which results in errors. The middleware ensures the query errors are returned as `ToolMessage` to the model to retry the queries instead of breaking the workflow.

In [ ]:
from typing import Callable

from langchain.agents.middleware import (AgentMiddleware, ModelRequest,
                                         ModelResponse)
from langchain.messages import ToolMessage
from neo4j.exceptions import ClientError


class RetryCypherMiddleware(AgentMiddleware):

    def wrap_tool_call(self, request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
        """Retry a Cypher tool call and convert database errors into a user-safe tool response.

        This middleware wrapper catches Neo4j ClientError exceptions raised by a Cypher tool
        execution and returns a standardized `ToolMessage` that preserves the original
        tool call ID while shielding end users from raw database error details.

        Args:
            request (ModelRequest): The model request containing the tool call context.
            handler (Callable[[ModelRequest], ModelResponse]): The wrapped tool handler.

        Returns:
            ModelResponse: The successful tool response, or a `ToolMessage` containing
            a human-friendly error message on failure.
        """

        try:
            return handler(request)

        except ClientError as err:
            return ToolMessage(
                content=f"Tool error: Please check your input and try again. ({str(err)})",
                tool_call_id=request.tool_call["id"]
            )

    async def awrap_tool_call(self, request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
        """Retry a Cypher tool call and convert database errors into a user-safe tool response.

        This middleware wrapper catches Neo4j ClientError exceptions raised by a Cypher tool
        execution and returns a standardized `ToolMessage` that preserves the original
        tool call ID while shielding end users from raw database error details.

        Args:
            request (ModelRequest): The model request containing the tool call context.
            handler (Callable[[ModelRequest], ModelResponse]): The wrapped tool handler.

        Returns:
            ModelResponse: The successful tool response, or a `ToolMessage` containing
            a human-friendly error message on failure.
        """

        try:
            return await handler(request)

        except ClientError as err:
            return ToolMessage(
                content=f"Tool error: Please check your input and try again. ({str(err)})",
                tool_call_id=request.tool_call["id"]
            )

Middleware hook that checks for truncated model output.

When using HuggingFace Inference API, output tokens are capped at 512 which prematurely cuts of the model response. This hook ensures model is prompted again to continue generating the output, which utlimately returns the complete output.

Do note that this behaviour has only been observed when using models loading from `HuggingFaceEndpoint`. For other providors, this middleware hook is not required.

In [ ]:
from typing import Any

from langchain.agents.middleware import AgentState, after_model, hook_config
from langchain.messages import HumanMessage
from langgraph.runtime import Runtime


@after_model
@hook_config(can_jump_to=["model"])
def check_truncated_output(state: AgentState, runtime: Runtime) -> dict[Any] | None:
    """
    Middleware hook to check for truncated model output
    If output is truncated, loops back to model node and asks for remaining output

    Args:
        state (AgentState): Graph state object to access state attributes
        runtime (Runtime): Runtime object to access runtime context

    Returns:
        Dictionary specifying which node to jump to incase of truncated output otherwise returns `None`
    """

    last_message = state["messages"][-1]

    if last_message.response_metadata["finish_reason"] == "length":
        return {
            "messages": [HumanMessage("Continue from precisely where you left off.")],
            "jump_to": "model"
        }
    return None

# Workflow Execution

Loads the configurations and starts the pipeline execution.

Create the graph database instance

In [ ]:
# Get memgraph database instance
db = connect_to_memgraph()

Load the required models

In [ ]:
# load the models
# hf_model = load_model_from_hf("deepseek-ai/DeepSeek-V3.2")
novita_model = load_novita_model("deepseek/deepseek-v3.2")
anthropic_model = load_anthropic_model("claude-opus-4-6")
openai_model = load_openai_model("gpt-5.1")

Create the agents. Three agents are created covering two categories
* Open Access Models
* Frontier Models

In [ ]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy

# instantiate hugging face agent
# hf_agent = create_agent(
#     hf_model,
#     get_memgraph_tools(db, hf_model),
#     system_prompt=SYSTEM_MSG,
#     middleware=[check_truncated_output, retry_cypher]
# ).with_config({"run_name": "HuggingFaceAgent"})

# instantiate novita agent
novita_agent = create_agent(
    novita_model,
    get_memgraph_tools(db, novita_model),
    system_prompt=SYSTEM_MSG,
    # response_format=RiskAssessment,
    middleware=[RetryCypherMiddleware()]
).with_config({
    "run_name": "NovitaAgent",
    "tags": ["novita", "deepseek"]
})

# instantiate anthropic agent
anthropic_agent = create_agent(
    anthropic_model,
    get_memgraph_tools(db, anthropic_model),
    system_prompt=SYSTEM_MSG,
    # response_format=ProviderStrategy(RiskAssessment)
).with_config({
    "run_name": "AnthropicAgent",
    "tags": ["anthropic", "opus-4.6"]
    })

# instantiate openai agent
openai_agent = create_agent(
    openai_model,
    get_memgraph_tools(db, openai_model),
    system_prompt=SYSTEM_MSG
    # response_format=ProviderStrategy(RiskAssessment)
).with_config({
    "run_name": "OpenAIAgent",
    "tags": ["openai", "gpt-5"]
    })

Display the agent stategraphs

In [ ]:
from IPython.display import Image, display

# display the stategraphs
display(Image(novita_agent.get_graph().draw_mermaid_png()))
display(Image(anthropic_agent.get_graph().draw_mermaid_png()))
display(Image(openai_agent.get_graph().draw_mermaid_png()))

Wrap the agent objects inside a `RunnableParallel` instance for concurrent execution of user queries

In [ ]:
from langchain_core.runnables import RunnableParallel

parallel_agents = RunnableParallel(
    novita=novita_agent,
    anthropic=anthropic_agent,
    openai=openai_agent,
)

Define input messages

In [ ]:
from langchain.messages import HumanMessage

user_message = {
    "messages": [
        HumanMessage("What is the risk posed by CVE-2025-67725?")
    ]
}

user_message1 = {
    "messages": [
        HumanMessage("Tell me a joke.")
    ]
}

user_message2 = {
    "messages": [
        HumanMessage("When did the Roman Empire Fall")
    ]
}

### Run the agents

Run a single agent with a user query

In [ ]:
result = novita_agent.invoke(user_message)

Run a single user query against multiple agents concurrently by executing on the `RunnableParallel`

In [ ]:
results = parallel_agents.invoke(user_message)

Run the user request in batches using `abatch` method. This executes agentic workflow for each user query concurrently 

In [ ]:
results = parallel_agents.abatch(
    inputs=[user_message1, user_message2],
    config={"max_concurrency": 7}
)

In [ ]:
structured_anthropic_model = anthropic_model.with_structured_output(RiskAssessment)

In [ ]:
risk_assessment = structured_anthropic_model.invoke(
    f"Given the vulnerability analysis, strcuture it according to the RiskAssessment schema: \n\n {final_text}"
)